# English Tutor PoC

Grammar-correction agent for English, modeled on `GermanTutorPoC.ipynb`. Grounded in
**New Round-Up 5 – English Grammar Practice** (Evans & Dooley, Pearson), an
intermediate-level reference covering Present/Past/Future Forms, Infinitive/-ing/Participles,
Modal Verbs, The Passive, Conditionals/Wishes, Clauses, Reported Speech, Nouns/Articles,
the Causative Form, Adjectives/Adverbs/Comparisons, Demonstratives/Pronouns/Possessives/
Quantifiers, Prepositions, and Questions & Answers.

In [1]:
from pydantic_ai import Agent
from pydantic_ai.models.openai import OpenAIModel
from pydantic_ai.providers.openai import OpenAIProvider

## 1. Build a searchable reference from the New Round-Up 5 PDF

We extract the PDF once into overlapping text chunks and cache them to JSON
(`new_round_up_5_reference.json`) next to this notebook, so re-running the notebook
doesn't require re-parsing the 200+ page PDF every time.

In [ ]:
import json
from pathlib import Path

PDF_PATH = Path("data/New-Round-Up-5.pdf")
REFERENCE_PATH = Path("new_round_up_5_reference.json")

CHUNK_SIZE = 1200
CHUNK_OVERLAP = 200


def _clean_page(text: str) -> str:
    lines = [l for l in text.splitlines() if l.strip() and "hasanboy" not in l.lower()]
    return "\n".join(lines)


def build_reference(pdf_path: Path) -> dict:
    from pypdf import PdfReader

    reader = PdfReader(str(pdf_path))
    chunks = []
    for page_num, page in enumerate(reader.pages, start=1):
        text = _clean_page(page.extract_text() or "")
        if len(text) < 40:
            continue
        start = 0
        while start < len(text):
            end = start + CHUNK_SIZE
            chunks.append({"page": page_num, "text": text[start:end]})
            if end >= len(text):
                break
            start = end - CHUNK_OVERLAP
    return {
        "source": pdf_path.name,
        "title": "New Round-Up 5 - English Grammar Practice (Evans & Dooley, Pearson)",
        "level": "intermediate",
        "chunks": chunks,
    }


if REFERENCE_PATH.exists():
    reference = json.loads(REFERENCE_PATH.read_text())
else:
    reference = build_reference(PDF_PATH)
    REFERENCE_PATH.write_text(json.dumps(reference, ensure_ascii=False))

print(f"Loaded {len(reference['chunks'])} reference chunks from {reference['source']}")

Loaded 527 reference chunks from New-Round-Up-5.pdf


## 2. Lightweight keyword search over the reference

No embedding model required for the PoC — a keyword-overlap score over the cached
chunks is enough for the agent to ground its grammar explanations in the book's own
wording via the `grammar_reference_lookup` tool below.

In [3]:
import re
from collections import Counter

STOPWORDS = {"the", "a", "an", "of", "and", "in", "to", "for", "is", "are", "on", "with", "this", "that"}


def _tokenize(text: str) -> list[str]:
    return [w for w in re.findall(r"[a-zA-Z']+", text.lower()) if w not in STOPWORDS and len(w) > 2]


_chunk_tokens = [Counter(_tokenize(c["text"])) for c in reference["chunks"]]


def search_reference(query: str, top_k: int = 3) -> str:
    """Keyword search over the New Round-Up 5 reference chunks."""
    q_tokens = _tokenize(query)
    scored = [
        (sum(tokens.get(t, 0) for t in q_tokens), i)
        for i, tokens in enumerate(_chunk_tokens)
    ]
    scored = [s for s in scored if s[0] > 0]
    scored.sort(reverse=True)
    top = scored[:top_k]
    if not top:
        return "No matching section found in New Round-Up 5 for this query."
    results = []
    for _, i in top:
        chunk = reference["chunks"][i]
        results.append(f"[p.{chunk['page']}] {chunk['text'].strip()}")
    return "\n\n---\n\n".join(results)


print(search_reference("present perfect vs past simple", top_k=1))

[p.112] onths / years, etc. before
• When the reporting verb is in the past, the verb tenses change as follows:
Direct speech Reported speech
present simple
“Tom needs a new bike," Dad said.
past simple
Dad said Tom needed a new bike.
present continuous
“He is watching TV," she said.
past continuous
She said he was watching TV.
present perfect
“We has just left, ” she said.
past perfect
She said he had just left.
past simple
“He left an hour ago," she said.
past simple or past perfect
She said he (had) left an hour before.
past continuous
7 was surfing the Net at two o'clock yesterday," 
he said.
past continuous or past perfect continuous 
He said he was surfing / had been surfing the
Net at two o'clock the day before.
future
“He 'll be back in an hour," she said.
conditional
She said he would be back in an hour.
present perfect continuous
"I've been typing since morning, ” she said.
past perfect continuous
She said she had been typing since morning.
If the direct verb is already in th

In [4]:
# Ollama exposes an OpenAI-compatible API — point directly at it
model = OpenAIModel(
    model_name="gemma4:26b",
    provider=OpenAIProvider(base_url="http://localhost:11434/v1"),
)

print("Model configured:", model)

Model configured: OpenAIModel()


/var/folders/6k/3yb_sy2d4plffrxc253561_40000gp/T/ipykernel_31725/896140090.py:2: DeprecationWarning: `OpenAIModel` was renamed to `OpenAIChatModel` to clearly distinguish it from `OpenAIResponsesModel` which uses OpenAI's newer Responses API. Use that unless you're using an OpenAI Chat Completions-compatible API, or require a feature that the Responses API doesn't support yet like audio.
  model = OpenAIModel(


In [5]:
from pydantic import BaseModel


class GrammarSummary(BaseModel):
    summary: str
    key_points: list[str]
    confidence: float  # 0.0 - 1.0

In [6]:
agent = Agent(
    model=model,
    output_type=GrammarSummary,
    system_prompt=(
        "You are an English grammar tutor. "
        "Analyse the user text in English and return a structured grammar summary." \
        "Ground every explanation in 'New Round-Up 5 English Grammar Practice' (Evans & Dooley, Pearson), " \
        "an intermediate-level reference covering Present/Past/Future Forms, Infinitive/-ing form/Participles, " \
        "Modal Verbs, The Passive, Conditionals/Wishes, Clauses, Reported Speech, Nouns/Articles, the Causative Form, " \
        "Adjectives/Adverbs/Comparisons, Demonstratives/Pronouns/Possessives/Quantifiers, Prepositions, and Questions & Answers. " \
        "Before explaining a rule, call the grammar_reference_lookup tool with a short query naming the topic so your " \
        "explanation matches how the book presents it. " \
        "The grammar summary should include the violated rules with the explanation of the rules and the number of times " \
        "the rules are violated in the text." \
        "Each rule should be explained in a simple way and include an example of the error and the correction. But the example is not taken from the text." \
        "No indication where the errors in text appear should be given. " \
        "The user should correct the text by himself and learn from the grammar summary. " \
        "The user should input the corrected text to the agent." \
        "The agent should check the corrected text and return the grammar summary again. " \
        "In case the text is big (> 250 words), the agent should split the text into smaller parts and return the grammar summary for each part. " \
        "In case several errors obey the same rule, cluster them and show the violated rule only once in the summary. " \
        "All errors, and the rules they violate, need to be stored in the memory of the agent." \
        "The memory of the agent is stored in a vector database and can be accessed by the agent at any time. " \
        "The agent should stop producing the grammar summary if the user input is correct, if the user input is not in English, or if the user input is empty. " \
        "The agent should produce the grammar summary only if the user input is in English and contains grammar errors. " \
        "The agent must stop if the user input includes the word 'stop' or 'exit'. " \
        "The agent should stop creating a grammar summary after 5 iterations of user input and grammar summary on the same text. " \
        "The user text is: {input}" \
        "The output should be in English." \
    ),
)

In [7]:
@agent.tool_plain
def grammar_reference_lookup(query: str) -> str:
    """Look up how New Round-Up 5 explains a grammar topic (e.g. 'modal verbs must have to')."""
    return search_reference(query, top_k=2)


_agent_memory: list[dict] = []


@agent.tool_plain
def memory_search(query: str) -> str:
    """Search the agent's memory of previously seen errors and rules."""
    q_tokens = set(_tokenize(query))
    hits = [
        m for m in _agent_memory
        if q_tokens & set(_tokenize(m["rule"] + " " + m["error"]))
    ]
    if not hits:
        return f"No memory entries match query: '{query}'"
    return "\n".join(f"- rule: {m['rule']!r}, error: {m['error']!r}" for m in hits)


@agent.tool_plain
def store_errors_in_memory(errors: list[str]) -> str:
    """Store identified errors in the agent's memory. Each entry is 'rule: error'."""
    for e in errors:
        rule, _, error = e.partition(":")
        _agent_memory.append({"rule": rule.strip(), "error": error.strip() or rule.strip()})
    return f"Stored {len(errors)} error(s) in memory (total: {len(_agent_memory)})."

## 3. First iteration — run the agent on flawed intermediate-level English

The sample text below deliberately exercises several New Round-Up 5 chapters: Present
Perfect vs Past Simple, Past Continuous, Modal Verbs (`must`), and Nouns/Articles.

In [8]:
async def run_agent(query: str) -> GrammarSummary:
    result = await agent.run(query)
    return result.output

# Jupyter has a running event loop, so use await directly
sample_text = (
    "I have went to London last year and I seen a lot of interesting place. "
    "I must to visit the Big Ben because my friend told me it's beautiful monument. "
    "Yesterday, I am walking in park when it start raining."
)

output = await run_agent(sample_text)

print(output.model_dump_json(indent=2))

APIConnectionError: Connection error.

## 4. Iterative correction loop (multi-turn)

Mirrors the intended workflow: the learner submits a correction, the agent re-checks it
and the summary should shrink as fewer rules are violated.

In [ ]:
from pydantic_ai.messages import ModelMessagesTypeAdapter

history = []

async def chat(user_input: str) -> GrammarSummary:
    global history
    result = await agent.run(user_input, message_history=history)
    history += result.new_messages()
    return result.output

# Turn 1: original flawed text
r1 = await chat(sample_text)
print("Turn 1 summary:\n", r1.summary, "\n")

# Turn 2: the learner's attempted correction (still has a couple of slips)
corrected_text = (
    "I went to London last year and I saw a lot of interesting places. "
    "I must visit Big Ben because my friend told me it is a beautiful monument. "
    "Yesterday, I was walking in the park when it started raining."
)
r2 = await chat(corrected_text)
print("Turn 2 summary:\n", r2.summary)

Turn 1 summary:
 Grammar Summary for: 'I have went to London last year and I seen a lot of interesting place. I must to visit the Big Ben because my friend told me it's beautiful monument. Yesterday, I am walking in park when it start raining.'

1. Past Simple vs Present Perfect with time expressions (1 violation): Use the Past Simple for finished actions at a specific time (e.g., 'last year'). Example Error: I have eaten an apple yesterday. Correction: I ate an apple yesterday.

2. Past Simple Forms (2 violations): Remember irregular verbs (see -> saw) and regular '-ed' endings (walk -> walked). Example Error: He goed to the store. Correction: He went to the store.

3. Modal Verbs (1 violation): 'Must' is followed by the base form without 'to'. Example Error: You must to study. Correction: You must study.

4. Nouns and Articles (4 violations): Ensure correct pluralization and use articles for singular countable nouns. Example Error: I saw a dog in street. Correction: I saw a dog in th

Turn 2 summary:
 Great job! Your corrected text is grammatically perfect. All the previous errors regarding verb tenses, modal verbs, and articles have been resolved.
